In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np

import pybedtools
from pybedtools import BedTool


#For plotting
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

#For statistics
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import itertools

import re
from Bio import SeqIO
import ast # for safe eveal, for parsing some of the data
import math
# --- Shared helpers: const.py lives in analyses/common ---
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'common')))

#import importlib
#importlib.reload(const)

import const #to reload use import(importlib) and then importlib.reload(const)
from const import pos_active_ctrl_color,neg_active_ctrl_color,highlight_color,custom_cmap
from const import set_equal_plot_limits
from const import plot_color_pallete
from const import custom_cmap_bolder
from const import FONT_SIZE_small
const.set_plot_style()
import matplotlib.ticker as mtick

# --- Working directory (EDIT): used for relative .bed/.csv I/O in some notebooks ---
WORK_DIR = '/home/labs/davidgo/Collaboration/backup/humanMPRA/scripts/produce_paper_figures'
os.chdir(WORK_DIR)

output_path = '/home/labs/davidgo/Collaboration/humanMPRA/paper/raw_plots/fig_library_and_QC/'

# Parameters

In [ ]:
# Use CPM normalization?
cpm=True

# Use log scale in visualization?
logScale=False

min_DNA_counts = 0

# Figure 1 - hMPRA validation

### import and process all libraries

### Load annotations

In [ ]:
diff_activity = pd.read_csv(f'/home/labs/davidgo/Collaboration/humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv', 
                     header=0)
diff_activity = diff_activity.drop_duplicates(subset=["oligo"], keep = "first") #There are several HH oligos which are duplicated
diff_activity = diff_activity[#~(diff_activity['orientation_fix']=='fixed_in_L4')&
                           ~(diff_activity['oligo'].str.contains('SCREEN_EH'))&
                          ~(diff_activity['oligo'].str.contains('hh.missing.oligos')) ]



# create BED of diff_activity
diff_activity_bed_df = diff_activity[["chromosome","start","end","oligo"]]
diff_activity_bed_df.loc[:,"start"] = diff_activity_bed_df["start"].map(int) # already zero-based - Make sure with Simon!!
diff_activity_bed_df.loc[:,"end"] = diff_activity_bed_df["end"].map(int)
annotations_bed_df=diff_activity_bed_df.sort_values(['chromosome','start'], ascending = [True, True])
annotations_be = pybedtools.BedTool.from_dataframe(diff_activity_bed_df)

In [ ]:
diff_activity['minimum_DNA_counts'] = (
    diff_activity[['DNA_counts_raw_ancestral', 'DNA_counts_raw_derived']]
    .min(axis=1)
)

In [ ]:

# thresholds you want
thresholds = [0, 10, 20, 50, 100,150,500]

# container for the frequency tables
freq_list = []

for t in thresholds:
    df_sub = diff_activity[diff_activity["minimum_DNA_counts"] >= t]

    # count True, False, NaN
    counts = df_sub["differential_activity"].value_counts(dropna=False)

    # ensure all categories exist
    freq_list.append({
        "threshold": t,
        "True": counts.get(True, 0),
        "False": counts.get(False, 0),
        "NaN": counts.get(np.nan, 0)
    })

freq_df = pd.DataFrame(freq_list)
freq_df


In [ ]:
# set index for easy plotting
plot_df = freq_df.set_index("threshold")

plt.figure(figsize=(10,6))

# stacked bar chart
plot_df[["True","False","NaN"]].plot(
    kind="bar",
    stacked=True,
    figsize=(10,7),
    edgecolor="black"
)

plt.xlabel("Minimum DNA count threshold")
plt.ylabel("Frequency")
plt.title("Distribution of differential_activity\nat different minimum DNA count thresholds")
plt.legend(title="differential_activity")
plt.tight_layout()
plt.show()


## Filter based on DNA counts

In [ ]:
# 1. Create a new df which is a subset of the original
cols_to_keep = [
    'logFC_derived_vs_ancestral',
    'differential_activity_fdr',
    'differential_activity',
    'activity_ancestral',
    'activity_derived',
    'DNA_counts_raw_ancestral',
    'DNA_counts_raw_derived'
]

volcano_df = diff_activity[cols_to_keep].copy()

volcano_df['minimum_DNA_counts'] = (
    volcano_df[['DNA_counts_raw_ancestral', 'DNA_counts_raw_derived']]
    .min(axis=1)
)


# 2. Filter out all non active oligos
print(f' Active cCREs: {len(volcano_df)}')

eps = 1e-300
volcano_df['minus_log10_fdr'] = -np.log10(volcano_df['differential_activity_fdr'] + eps)

#volcano_df = volcano_df[(volcano_df['activity_ancestral']=='active')
#                        &(volcano_df['activity_derived']=='active')]

print(len(volcano_df))

# Optional: compute -log10(FDR) (add tiny epsilon to avoid log10(0))
volcano_df['minus_log10_fdr'] = -np.log10(volcano_df['differential_activity_fdr'])


minus_log10_fdr_max = 50   # X: max allowed FDR
lfc_max = 2.5    # Y: max allowed |logFC|

volcano_df = volcano_df[
    (volcano_df['minus_log10_fdr'] <= minus_log10_fdr_max) &
    (volcano_df['logFC_derived_vs_ancestral'].abs() <= lfc_max)
].copy()


In [ ]:
fig, ax = plt.subplots()

x = volcano_df['logFC_derived_vs_ancestral'].values
y = volcano_df['minus_log10_fdr'].values
z = np.log2(volcano_df['minimum_DNA_counts'].values)

hb = ax.hexbin(
    x, y,
    C=z,                      # value to aggregate within each hex
    reduce_C_function=np.mean, # or np.median
    gridsize=200,             # <-- higher = higher resolution (try 150–400)
    mincnt=1,                 # only draw hexbins with >=1 point
    cmap=custom_cmap_bolder,
    linewidths=0,
)

# Threshold lines
fdr_thresh = 0.05
ax.axhline(-np.log10(fdr_thresh), linestyle='--')
ax.axvline(0, linestyle='--')

ax.set_xlabel('logFC_derived_vs_ancestral')
ax.set_ylabel('-log10(FDR)')
ax.set_ylim(-5, 50)

cbar = plt.colorbar(hb, ax=ax, pad=0.02)
cbar.set_label('mean log2(minimum_DNA_counts)')

xticks = plt.xticks()[0]
yticks = plt.yticks()[0]
plt.xticks([xticks[0], xticks[-1]])
plt.yticks([yticks[0], yticks[-1]])

plt.tight_layout()
const.save_fig(plt, 'diff_activity_volcano_hexbin_by_min_dna_counts', output_path)
plt.show()


In [ ]:
fig, ax = plt.subplots()

# X/Y for volcano
x = volcano_df['logFC_derived_vs_ancestral'].values
y = volcano_df['minus_log10_fdr'].values

# Hexbin density
hb = ax.hexbin(
    x, y,
    gridsize=200,             # adjust for coarser/finer grid
    cmap=custom_cmap_bolder,  # your existing cmap
    mincnt=1,                 # only show hexes with at least 1 point
    bins='log',                # log-scale colors by count
    linewidths=0
)

# Threshold lines
fdr_thresh = 0.05
ax.axhline(-np.log10(fdr_thresh), linestyle='--')
ax.axvline(0, linestyle='--')

ax.set_xlabel('logFC derived vs ancestral')
ax.set_ylabel('-log10(FDR)')
ax.set_ylim(-5, 50)

# Optional colorbar for density
cbar = plt.colorbar(hb, ax=ax)
cbar.set_label('log10(count)')

xticks = plt.xticks()[0]
yticks = plt.yticks()[0]
plt.xticks([xticks[0], xticks[-1]])
plt.yticks([yticks[0], yticks[-1]])

plt.tight_layout()

const.save_fig(plt, 'full_diff_activity_hexbin_by_density', output_path)

plt.show()


In [ ]:
print(f"cCRE counts before filtering for minimum {min_DNA_counts} DNA counts in ancestral and derived (at least 5 in each): {len(diff_activity)}")
diff_activity = diff_activity[(diff_activity['DNA_counts_raw_derived']>min_DNA_counts) &
                               (diff_activity['DNA_counts_raw_ancestral']>min_DNA_counts)]
print(f"cCRE counts after filtering for minimum {min_DNA_counts} DNA counts in ancestral and derived (at least 5 in each): {len(diff_activity)}")


In [ ]:
# 1. Create a new df which is a subset of the original
cols_to_keep = [
    'logFC_derived_vs_ancestral',
    'differential_activity_fdr',
    'differential_activity',
    'activity_ancestral',
    'activity_derived',
    'DNA_counts_raw_ancestral',
    'DNA_counts_raw_derived'
]

volcano_df = diff_activity[cols_to_keep].copy()

volcano_df['minimum_DNA_counts'] = (
    volcano_df[['DNA_counts_raw_ancestral', 'DNA_counts_raw_derived']]
    .min(axis=1)
)


# 2. Filter out all non active oligos
print(f' Active cCREs: {len(volcano_df)}')

eps = 1e-300
volcano_df['minus_log10_fdr'] = -np.log10(volcano_df['differential_activity_fdr'] + eps)

#volcano_df = volcano_df[(volcano_df['activity_ancestral']=='active')
#                        &(volcano_df['activity_derived']=='active')]

print(len(volcano_df))

# Optional: compute -log10(FDR) (add tiny epsilon to avoid log10(0))
volcano_df['minus_log10_fdr'] = -np.log10(volcano_df['differential_activity_fdr'])


minus_log10_fdr_max = 50   # X: max allowed FDR
lfc_max = 2.5    # Y: max allowed |logFC|

volcano_df = volcano_df[
    (volcano_df['minus_log10_fdr'] <= minus_log10_fdr_max) &
    (volcano_df['logFC_derived_vs_ancestral'].abs() <= lfc_max)
].copy()


In [ ]:
fig, ax = plt.subplots()

# X/Y for volcano
x = volcano_df['logFC_derived_vs_ancestral'].values
y = volcano_df['minus_log10_fdr'].values

# Hexbin density
hb = ax.hexbin(
    x, y,
    gridsize=200,             # adjust for coarser/finer grid
    cmap=custom_cmap_bolder,  # your existing cmap
    mincnt=1,                 # only show hexes with at least 1 point
    bins='log',                # log-scale colors by count
    linewidths=0
)

# Threshold lines
fdr_thresh = 0.05
ax.axhline(-np.log10(fdr_thresh), linestyle='--')
ax.axvline(0, linestyle='--')

ax.set_xlabel('logFC_derived_vs_ancestral')
ax.set_ylabel('-log10(FDR)')
ax.set_ylim(-5, 50)

xticks = plt.xticks()[0]
yticks = plt.yticks()[0]
plt.xticks([xticks[0], xticks[-1]])
plt.yticks([yticks[0], yticks[-1]])

plt.tight_layout()

const.save_fig(plt, 'filtered_50_diff_activity_hexbin_by_density', output_path)

# Optional colorbar for density
cbar = plt.colorbar(hb, ax=ax)
cbar.set_label('log10(count)')

const.save_fig(plt, 'filtered_50_diff_activity_hexbin_by_density_w_bar', output_path)


plt.show()


# Validatio of differential expression

In [ ]:
libraries = [
    #"L3a3",
     "L1a1", "L1a2", "L1a3",
     "L2a1", "L2a2", "L2a3",
     "L3a1", "L3a2", "L3a3",
     "L4a1",
]

base_dir = "/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes"
rel_path = "output/mpranalyze_comparative/posctrl/mpranalyze_comp_res_filter_sorted.txt"

dfs = []
percents = []

for lib in libraries:
    positive_controls_dir = os.path.join(base_dir, lib, rel_path)
    df = pd.read_csv(positive_controls_dir, sep="\t")
    df["library"] = lib
    print(lib)

    successful_controls = (df["fdr"] < 0.05).sum()
    total_controls = len(df)

    pct = (successful_controls / total_controls) * 100 if total_controls else float("nan")
    percents.append(pct)

    print(f"{pct:.2f}%")
    dfs.append(df)

positive_controls = pd.concat(dfs, ignore_index=True)

avg_pct = sum(percents) / len(percents)
print(f"\nAverage across libraries: {avg_pct:.2f}%")



In [ ]:
print(sum(positive_controls['fdr']<0.05))
print(len(positive_controls))

In [ ]:
fig, ax = plt.subplots()

# X/Y for volcano
x = volcano_df['logFC_derived_vs_ancestral'].values
y = volcano_df['minus_log10_fdr'].values
z = np.log2(volcano_df['minimum_DNA_counts'].values)
# Hexbin density

hb = ax.hexbin(
    x, y,
    gridsize=100,
    mincnt=1,
    linewidths=0,
    edgecolors="none",
)

# force uniform styling (overrides colormap behavior)
hb.set_array(None)              # <-- this is the key if it's still colored
hb.set_facecolor("lightgray")
hb.set_edgecolor("none")
hb.set_alpha(1)


# hb = ax.scatter(
#     x, y,
#     c='lightgray',
#     alpha = 1,           
#     cmap=custom_cmap_bolder,  # your existing cmap
# )

positive_controls["minus_log10_fdr"] = -np.log10(positive_controls["fdr"].astype(float))
positive_controls_capped = positive_controls[positive_controls['minus_log10_fdr']<=50]

hb = ax.hexbin(
    positive_controls_capped["logFC"].values,
    positive_controls_capped["minus_log10_fdr"].values,
    C=z,                      # value to aggregate within each hex
    reduce_C_function=np.mean, # or np.median
    gridsize=100,             # <-- higher = higher resolution (try 150–400)
    mincnt=1,                 # only draw hexbins with >=1 point
    cmap="Greens",
    linewidths=0,
)

# Add positive controls for differential expression
# ax.scatter(
#     positive_controls["logFC"].values,
#     positive_controls["minus_log10_fdr"].values,
#     marker="o",
#     s=45,
#     c = 'green',
#     edgecolors=None,
#     linewidths=0.6,
#     alpha=1,
#     label="Positive controls"
# )



# Threshold lines
fdr_thresh = 0.05
ax.axhline(-np.log10(fdr_thresh), linestyle='--')
ax.axvline(0, linestyle='--')

ax.set_xlabel('logFC_derived_vs_ancestral')
ax.set_ylabel('-log10(FDR)')
ax.set_ylim(-1, 50)


const.save_fig(plt, 'diff_activity_controls_scatter_plot', output_path)

# Optional colorbar for density
cbar = plt.colorbar(hb, ax=ax)
cbar.set_label('log10(count)')

const.save_fig(plt, 'diff_activity_controls_scatter_plot_w_bar', output_path)


plt.tight_layout()
plt.show()


In [ ]:
hb = ax.scatter(
    x, y,
    c=z,
    alpha = 0.1,           
    cmap=custom_cmap_bolder,  # your existing cmap
)

In [ ]:
positive_controls

In [ ]:
# List of libraries
libraries = ["L3a2"
    #"L1a1", "L1a2", "L1a3",
    #"L2a1", "L2a2", "L2a3",
    #"L3a1", "L3a2", "L3a3",
    #"L4a1"
]

base_dir = r"/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes"

dfs = []

for lib in libraries:
    csv_path = os.path.join(
        base_dir,
        lib,
        "output",
        "activity_after_filter",
        "comb_df_adjusted_fdr.csv"
    )

    # Read CSV
    df = pd.read_csv(csv_path)

    # Keep only rows where "oligo" contains "ctrl"
    df_ctrl = df[
    df["oligo"].str.contains("Ctrl", na=False, case=False) |
    df["oligo"].str.contains("scramble", na=False, case=False)
        ].copy()
    
    # Optional: add a column to remember which library it came from
    df_ctrl["library"] = lib

    dfs.append(df_ctrl)

# Combine all control oligos from all libraries into one df
all_ctrl_oligos = pd.concat(dfs, ignore_index=True)


In [ ]:
control_types = [
    "NegCtrl_active_not_diff_ESCs+Osteoblasts+NPCs",
    "NegCtrl_non_SCREEN",
    "scrambled_control",
    "PosCtrl_diff_ESCs_Weiss_seq11687_derived_a1_L4",
    "NegCtrl_not_active_ESCs+Osteoblasts+NPCs",
    "PosCtrl_diff_ESCs+Osteoblasts+NPCs",
    "PosCtrl_chondrocyte_active",
    "PosCtrl_diff_NPCs_Weiss",
    "PosCtrl_diff_ESCs+NPCs",
    "PosCtrl_diff_ESCs_Weiss",
    "PosCtrl_diff_ESCs+Osteoblasts",
    "PosCtrl_diff_Osteoblasts+NPCs",
    "PosCtrl_diff_Osteoblasts",
    "PosCtrl_neuron_active",
    "PosCtrl_osteoblast_active",
]

ctrl_dict = {}

for ctype in control_types:
    mask = all_ctrl_oligos["oligo"].str.contains(ctype, regex=False)
    ctrl_dict[ctype] = all_ctrl_oligos[mask].copy()

# Optional: if you want a df of any remaining controls that didn't match:
unmatched_mask = ~pd.concat(
    [all_ctrl_oligos["oligo"].str.contains(ct, na=False, regex=False) for ct in control_types],
    axis=1
).any(axis=1)

ctrl_dict["other_ctrls"] = all_ctrl_oligos[unmatched_mask].copy()

to_combine = [
    "PosCtrl_chondrocyte_active",
    "PosCtrl_osteoblast_active",
    "NegCtrl_active_not_diff_ESCs+Osteoblasts+NPCs"
]

ctrl_dict["all_activity_pos_ctrl"] = (
    pd.concat([ctrl_dict[k] for k in to_combine], ignore_index=True)
)

#Add the tested cCREs
# Add tested cCREs as a new entry in the control dictionary
# (here we only keep mad.score since that's what we plot)
ctrl_dict["tested_cCREs"] = full_activity_df[["mad.score"]]



## Correlation between libraries

In [ ]:
controls_corr_dir = "/home/labs/davidgo/Collaboration/humanMPRA/additional/analyze_controls/chondrocytes/controls_df.csv"
ctrls_corr_df = pd.read_csv(controls_corr_dir)
ctrls_corr_df = ctrls_corr_df.loc[:, ~ctrls_corr_df.columns.str.contains("L4a2", na=False)]
alpha = ctrls_corr_df.filter(regex='alpha')

In [ ]:
active_patterns = [
    "PosCtrl_chondrocyte_active",
    "PosCtrl_osteoblast_active",
    "NegCtrl_active_not_diff",
]

negative_patterns = [
    "scrambled_control",
    "NegCtrl_not_active",
    "NegCtrl_non_SCREEN",
]

oligos = ctrls_corr_df["oligo"].fillna("")

active_mask = pd.Series(False, index=ctrls_corr_df.index)
for p in active_patterns:
    active_mask |= oligos.str.contains(p, regex=False)

negative_mask = pd.Series(False, index=ctrls_corr_df.index)
for p in negative_patterns:
    negative_mask |= oligos.str.contains(p, regex=False)

active_ctrls_corr_df = ctrls_corr_df[active_mask].copy()
negative_ctrls_corr_df = ctrls_corr_df[negative_mask].copy()

positive_ctrls_alpha = active_ctrls_corr_df.filter(regex='alpha')
negative_ctrls_alpha = negative_ctrls_corr_df.filter(regex='alpha')

positive_ctrls_corr_alpha = positive_ctrls_alpha.corr()
negative_ctrls_corr_alpha = negative_ctrls_alpha.corr()

negative_ctrls_corrc_alpha = negative_ctrls_corr_alpha.round(2)
positive_ctrls_corr_alpha = positive_ctrls_corr_alpha.round(2)



In [ ]:
active_ctrls_corr_df

In [ ]:
output_path

In [ ]:
plt.clf()
sns.heatmap(positive_ctrls_corr_alpha, cmap="Blues", annot=True)
const.save_fig(plt, 'pos_ctrl_correlation_across_libraries', output_path)

plt.show()

In [ ]:
plt.clf()
sns.heatmap(negative_ctrls_corrc_alpha, cmap="Blues", annot=True)
plt.show()

In [ ]:
corr_alpha = alpha.corr()
plt.clf()
sns.heatmap(corr_alpha, cmap="Blues", annot=True)
plt.show()

# GC content in the library

In [ ]:
# load library design

library = pd.read_excel(f'/home/labs/davidgo/Collaboration/humanMPRA_raw/sequences.xlsx', 
                    sheet_name = 'oligos',
                    #usecols=lambda x: x not in ["derived_seq_with_adapters", "ancestral_seq_with_adapters"],
                     header=0)

# filter for SCREEN oligos
print(f'rows before filtering for SCREEN: {len(library)}')
library = library[library['source'].str.contains('SCREEN')]
print(f'rows after filtering for SCREEN: {len(library)}')
print(f'oligo counts after filtering for SCREEN: {len(library)*2}')

# load L4 library design
library_L4 = pd.read_excel(
    "/home/labs/davidgo/Collaboration/humanMPRA_raw/L4/L4_v5.xlsx",
    sheet_name="test_oligos",
    #usecols=lambda x: x not in ["derived_sequence_with_adapters", "ancestral_seq_with_adapters"],
    header=0
)


In [ ]:

#change derived_sequence_with_adapters name to derived_seq_with_adapters
library_L4 = library_L4.rename(columns={"derived_sequence_with_adapters": "derived_seq_with_adapters"})
library_L4 = library_L4.rename(columns={"ancestral_sequence_with_adapters": "ancestral_seq_with_adapters"})

# filter for SCREEN oligos in L4
library_L4_filtered = library_L4.copy()
library_L4_filtered = library_L4_filtered[(library_L4_filtered['adapter']=='a1')&
                                          (library_L4_filtered['name'].str.contains('SCREEN'))&
                                          (library_L4_filtered['class'].isin(['CGI.fix','promoter.orientation.fix']))]
print(f'oligo counts in L4: {len(library_L4_filtered)*2}')

# check how many of the oligos I removed from the original library design are now fixed in L4a1
fixed_variants = library_L4_filtered['variants_genomic']
fixed_oligos = sum(library['variants_genomic'].isin(fixed_variants))
print(f'oligos fixed in L4a1 which I removed from the original library design:{fixed_oligos}')

filtered_library = library[~library['variants_genomic'].isin(fixed_variants)]


# columns shared by both
common_cols = filtered_library.columns.intersection(library_L4_filtered.columns)

# concatenate, keeping only shared columns (aligned + same order)
final_oligos = pd.concat([library_L4_filtered[common_cols],filtered_library[common_cols]], ignore_index=True)

final_oligos = final_oligos.drop_duplicates(
    subset="variants_genomic",
    keep="first"
)



In [ ]:

all_variants = (
    final_oligos["variants_genomic"]
      .dropna()
      .str.split(";")
      .explode()
      .tolist()
)
len(set(all_variants))



## Calculate GC content

In [ ]:
def GC_calc(seq):
    C = seq.count("C") + seq.count("c")
    G = seq.count("G") + seq.count("g")
    perc = (C + G) / len(seq)
    return perc

#make sequence capitalized
final_oligos['derived_seq_with_adapters'] = final_oligos['derived_seq_with_adapters'].str.upper()
final_oligos['ancestral_seq_with_adapters'] = final_oligos['ancestral_seq_with_adapters'].str.upper()

In [ ]:
#create a new vector of GC ocntent in all sequences
final_oligos['GC_content_derived'] = final_oligos['derived_seq_with_adapters'].apply(GC_calc)
final_oligos['GC_content_ancestral'] = final_oligos['ancestral_seq_with_adapters'].apply(GC_calc)
GC_content = pd.concat([final_oligos['GC_content_derived'], final_oligos['GC_content_ancestral']], ignore_index=True)
plt.hist(GC_content, bins=20)

In [ ]:
# read this fasta file: "\\data.wexac.weizmann.ac.il\davidgo\Collaboration\humanMPRA\oligo_fasta\L1L2L3L4a1.fasta"
fasta_path = "/home/labs/davidgo/Collaboration/humanMPRA/oligo_fasta/L1L2L3L4a1.fasta"
sequences = list(SeqIO.parse(fasta_path, "fasta")) 

def GC_calc(seq):
    C = seq.count("C") + seq.count("c")
    G = seq.count("G") + seq.count("g")
    perc = (C + G) / len(seq)
    return perc


In [ ]:

# Create a dataframe with sequences and their GC content
gc_data = []
for seq in sequences:
    seq_str = str(seq.seq)
    gc_percent = GC_calc(seq_str)
    gc_data.append({
        'sequence_id': seq.id,
        'gc_content': gc_percent
    })
gc_df = pd.DataFrame(gc_data)


In [ ]:
#read cCRE molecule counts
cCRE_association_counts_path = "/home/labs/davidgo/Collaboration/humanMPRA/neurons/combined_files/cCRE_association_counts.csv"
cCRE_association_counts = pd.read_csv(cCRE_association_counts_path)

In [ ]:
# Merge gc_df with cCRE_association_counts on sequence_id and cCRE
final_counts_df = gc_df.merge(
    cCRE_association_counts,
    left_on='sequence_id',
    right_on='cCRE',
    how='inner'
)

# Rename gc_content to gc for easier reference
final_counts_df = final_counts_df.rename(columns={'gc_content': 'gc'})

In [ ]:
final_counts_df

In [ ]:
def PCR_bias_GC_plot(final_counts_df):
    gc_bins = pd.cut(final_counts_df["gc"], bins=np.arange(0, 1.01, 0.05), duplicates="drop")
    final_counts_df["gc_bin"] = gc_bins
    bin_sizes = final_counts_df.reset_index().groupby("gc_bin")["index"].nunique()
    bin_df = pd.DataFrame(data={"gc_bin": bin_sizes.index, "bin_size": bin_sizes.values})

    bin_df["gc_bin_center"] = bin_df["gc_bin"].apply(lambda x: (float(x.left) + float(x.right)) / 2)
    bin_intervals = bin_df["gc_bin"].cat.categories
    bin_edges = [i.left for i in bin_intervals] + [bin_intervals[-1].right]

    boxplot_df = final_counts_df.copy()
    boxplot_df["gc_bin_center"] = boxplot_df["gc_bin"].apply(lambda x: (float(x.left) + float(x.right)) / 2)
    boxplot_groups = boxplot_df.groupby("gc_bin_center")["association_count"].apply(list)

    gc_summary = boxplot_df.groupby("gc_bin_center", observed=False)["association_count"].agg(["count", "median"]).reset_index()

    bin_width_dict = {(i.left + i.right) / 2: (i.right - i.left) / 2 for i in bin_intervals}
    widths_filtered = [bin_width_dict.get(pos, 0.5) for pos in boxplot_groups.index]

    f, ax_hist = plt.subplots()
    ax_hist.boxplot(
        x=boxplot_groups.values,
        positions=boxplot_groups.index,
        showfliers=False,
        widths=widths_filtered,
        patch_artist=True,
        boxprops=dict(facecolor=plot_color_pallete["read"]),
        medianprops=dict(color="black", linewidth=1),
    )
    ax_hist.set_xticks([bin_edges[0], bin_edges[-1]])
    ax_hist.set_xlabel("GC content")
    ax_hist.set_ylabel("Number of reads")
    ax_hist.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax_hist.set_xlim(bin_edges[0], bin_edges[-1])
    ax_hist.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax2 = ax_hist.twinx()
    ax2.plot(gc_summary["gc_bin_center"], gc_summary["count"], color=plot_color_pallete["cCRE"], marker="o", label="cCRE count")
    ax2.set_ylabel("Number of cCREs")
    ax2.yaxis.label.set_color(plot_color_pallete["cCRE"])
    ax_hist.yaxis.label.set_color(plot_color_pallete["read"])
    ax_hist.tick_params(axis="y", colors=plot_color_pallete["read"])
    ax2.tick_params(axis="y", colors=plot_color_pallete["cCRE"])
    ax_hist.spines["right"].set_visible(True)
    f.set_size_inches(8, 6)
    print("PCR_bias_GC DONE")
    plt.tight_layout()
    const.save_fig(plt, 'GC_content', output_path)

    plt.show()

In [ ]:
PCR_bias_GC_plot(final_counts_df)



## Produce RNA/DNA MAD score correlaiton fig

In [ ]:
full_activity_df = pd.read_csv('/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes/quantitative_analysis_combined/comb_df_combined_fdr.csv')


In [ ]:
# Keep only rows where the 'oligo' column contains "SCREEN"
print(f"oligos before filtering for SCREEN overlap:{len(full_activity_df)}")
full_activity_df = full_activity_df[full_activity_df["oligo"].str.contains("SCREEN", na=False)]
print(f"oligos after filtering for SCREEN overlap:{len(full_activity_df)}")
full_activity_df =  full_activity_df[~full_activity_df["oligo"].str.contains("Ctrl", na=False)]
print(f"oligos after filtering for non-controls:{len(full_activity_df)}")

min_DNA_counts = 50
if min_DNA_counts>0:
    full_activity_df =  full_activity_df[(full_activity_df['DNA_rep_comb']>min_DNA_counts)]
    print(f"oligos after filtering for oligos with over 10 DNA counts (combined for all replicates):{len(full_activity_df)}")



# remove oligos which were later removed
full_activity_df = full_activity_df[~(full_activity_df['orientation_fix']=='fixed_in_L4')&
                           ~(full_activity_df['oligo'].str.contains('SCREEN_EH'))&
                          ~(full_activity_df['oligo'].str.contains('hh.missing.oligos')) ]
print(f"oligos after filtering out oligos which were later remoed due to L4a1 (eg orientation fix):{len(full_activity_df)}")

#if min_barcodes>0:
#    full_activity_df =  full_activity_df[(full_activity_df['bcs_DNA_rep1']>10) &(full_activity_df['bcs_DNA_rep2']>10)]
#    print(f"oligos after filtering for oligos with over 10 barcodesin both reps 1 and 2:{len(full_activity_df)}")


In [ ]:
full_activity_df

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import spearmanr


mask = full_activity_df["mad.score"]<200

x = full_activity_df["ratio_log_rep_comb"][mask]
y = np.log2(full_activity_df["mad.score"][mask])

valid = ~x.isna() & ~y.isna()
x = x[valid]
y = y[valid]

rho, pval = spearmanr(x, y)


fig, ax = plt.subplots(figsize=(6, 6))

# Hexbin density plot instead of scatter
hb = ax.hexbin(
    x, y,
    gridsize=200,              # tweak for smoother/coarser bins
    cmap=custom_cmap_bolder,  # same colormap as before
    mincnt=1,                 # only show bins with at least 1 point
    bins='log',                # log10 of counts
    linewidth = 0
)

# Dashed reference lines at x = 0 and y = 0
#ax.axhline(0, linestyle="--", linewidth=1)
#ax.axvline(0, linestyle="--", linewidth=1)

ax.set_xlabel("log(RNA DNA ratio)")
ax.set_ylabel("log(MAD score)")

# Add Spearman's rho and p-value to plot
ax.text(
    0.05, 0.95,
    f"Spearman's rho = {rho:.3f}\nP = {pval:.2e}\nN = {len(x)}",
    #f"Pearson r = {r:.3f}",
    transform=ax.transAxes,
    va="top",
    ha="left"
)

plt.tight_layout()

const.save_fig(plt, 'statistic_RNA_DNA_corr', output_path)

plt.show()

## Find specific oligos example

In [ ]:
a3L3_df = full_activity_df[full_activity_df['oligo'].str.contains('_a3_L3')]
rep1_dna_sum = a3L3_df['DNA_rep1'].sum()
rep1_rna_sum = a3L3_df['RNA_rep1'].sum()

seq_name = 'seq_343252_chr1:32326432-32326701_SCREEN_ancestral_a3_L3'
a3L3_df = a3L3_df[a3L3_df['oligo'] == seq_name]


dna_count = (a3L3_df['DNA_rep1'].values[0]/rep1_dna_sum)*1_000_000
rna_count = (a3L3_df['RNA_rep1'].values[0]/rep1_rna_sum)*1_000_000

print(f"rep1: DNA count per million: {dna_count:.2f}, RNA count per million: {rna_count:.2f}")
print(rna_count/dna_count)
np.log2(rna_count/dna_count)

## Load RNA_DNA ratio df for L3a3

In [ ]:
seq_name = 'seq_350395_chr5:67595471-67595740_SCREEN_derived_a3_L3'

In [ ]:
ratio_df_L3a3 = pd.read_csv("/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes/L3a3/output/activity_after_filter/ratio_after_filter.csv")
ratio_df_L3a3 = ratio_df_L3a3[ratio_df_L3a3['oligo_column']==seq_name]

In [ ]:
ratio_df_L3a3

In [ ]:
a3L3_df = full_activity_df[full_activity_df['oligo'].str.contains('_a3_L3')]
a3L3_df = a3L3_df[a3L3_df['oligo'] == seq_name]

dna_count = (ratio_df_L3a3['DNA_rep1'].values[0]/rep1_dna_sum)*1_000_000
rna_count = (ratio_df_L3a3['RNA_rep1'].values[0]/rep1_rna_sum)*1_000_000



dna_count = ((dna_count)/rep1_dna_sum)*1_000_000
rna_count = ((rna_count)/rep1_rna_sum)*1_000_000

print(f"rep1: DNA count per million: {dna_count:.2f}, RNA count per million: {rna_count:.2f}")

In [ ]:
a3L3_df

In [ ]:
#filter for pval.mad>0.1 and then sort descending by ratio_log_rep1 #also only rows with a3_L3
query = full_activity_df[full_activity_df['pval.mad']>0.1]
query = query[query['oligo'].str.contains('_a3_L3')]
query = query[query['ratio_log_rep_comb']>1]

query = query.sort_values(by='ratio_log_rep1', ascending=False)